# Comparación de aceros en el recocido de rollos (horno tipo campana)

Simula los aceros de `data/materials/propiedades_termicas_aceros.csv` con el **mismo** horno, gas, malla, geometría de rollo y paso de tiempo, y los compara con las mismas unidades (horas y °C).

## Imports

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from heattransfer.simulator import Simulator
from heattransfer.models.furnace import Furnace
from heattransfer.models.coil import Coil
from heattransfer.models.protection_gas import ProtectionGas
from heattransfer.models.steel_catalog import load_steel_catalog, build_material
from heattransfer.analysis.analyzer import Analyzer, SteelComparison

## Configuración

Del CSV salen densidad, calor específico, conductividad y las dimensiones del rollo. Lo que el CSV no trae (emisividad, dureza, rugosidad, espesor de tira, óxido, gas, flujo y perfil del horno) se toma igual que en `main.ipynb` para ambos aceros, de modo que la única diferencia entre las simulaciones es el material.

In [2]:
catalog = load_steel_catalog("./data/materials/propiedades_termicas_aceros.csv")

gas = ProtectionGas(conductivity=0.1805, viscosity=8.9e-6, density=0.0819)

# Perfil de temperatura del gas del horno (igual para todos los aceros)
gas_temp = np.genfromtxt("./data/experimental/Thermocouples/1#.csv", delimiter=",", skip_header=1)
gas_temp = gas_temp[~np.isnan(gas_temp).any(axis=1)]

simulators, furnaces = {}, {}
for name, properties in catalog["steels"].items():
    material = build_material(
        properties,
        emissivity=0.08554,
        hardness=1133.86,
        roughness=3.2215e-6,
    )
    coil = Coil(
        material=material,
        **catalog["coil"],              # radios interior/exterior y ancho del CSV
        strip_thickness=1.48e-3,
        point_separation=0.05,
        initial_temperature=(273.15 + 55),
        surface_slope=0.1,
        oxide_thickness=10e-6,
        oxide_conductivity=3.0,
        emulsion_correction=0.8,
        geometry="cylindrical",         # "planar" reproduce el esquema anterior
    )
    furnaces[name] = Furnace(
        temperature=gas_temp,
        height=5.59,                    # (Zhang et. al)
        radius=1.03,                    # (Zhang et. al)
        coil_sep_height=0.10,           # (Zhang et. al)
        coils=[coil],
        gas=gas,
        gas_flow=19.4,                  # (Zhang et. al)
        vortex_enhancement=1.0,         # (Zhang et. al)
        radiation_enhancement=1.1,
    )
    # dt = 30 s: con k = 89 W/mK la estabilidad axial exige dt < dz²/(2·alpha) ≈ 57 s
    simulators[name] = Simulator(furnace=furnaces[name], time_step=30.0,
                                 simulation_time=furnaces[name].duration)
list(simulators)

['SAE J403', 'SAE J1392']

## Simulate
Activar `simulator.run()` para cada acero (≈ 20 s cada uno).

In [ ]:
results = {name: simulator.run() for name, simulator in simulators.items()}

## Análisis

### Settings

`target_c` es la temperatura objetivo. El instante en que un acero la alcanza es el primero en que su **punto frío** (la zona más lenta de la sección) llega a ese valor. Para seguir un punto concreto, usar `point=(r, z)` en metros.

In [ ]:
target_c = 650.0

comparison = SteelComparison(
    {name: Analyzer(results=results[name], simulator=simulators[name], furnace=furnaces[name])
     for name in simulators},
    point=None,
)

times = comparison.target_times_h(target_c)
names = comparison.names
print(f"Objetivo: {target_c:g} °C")
for name in names:
    print(f"  {name}: {times[name]:.2f} h")
print(f"  Δt = t({names[0]}) − t({names[1]}) = {times[names[0]] - times[names[1]]:+.2f} h")

### Plot — Curvas superpuestas

Temperatura del punto frío de cada acero contra el tiempo, con el gas del horno como referencia. Los rombos marcan el instante en que cada acero alcanza el objetivo y la flecha muestra la diferencia de tiempos.

In [ ]:
overlay_fig = comparison.plot_overlay(target_c, show_hot_spot=False)
overlay_fig.show()

### Plot — Efecto de la temperatura objetivo

Tiempo necesario para cada objetivo (izquierda) y diferencia entre aceros (derecha). Positivo: el segundo acero llega antes.

In [ ]:
sweep_fig = comparison.plot_target_sweep()   # objetivos de 150 °C al máximo alcanzable por ambos, cada 25 °C
sweep_fig.show()

In [ ]:
for row in comparison.target_table():
    print(f"{row['objetivo_C']:5.0f} °C | "
          + " | ".join(f"{n}: {row[f't_{n}_h']:6.2f} h" for n in names)
          + f" | Δt = {row['diferencia_h']:+6.2f} h ({row['diferencia_pct']:.1f} %)")

## Heatmap (animación)

Solo animación: pulsa ▶ Reproducir o arrastra el selector de tiempo. Una sección por acero, una al lado de la otra, suavizada con splines cúbicos. La escala de color va de la temperatura inicial a la temperatura objetivo y la animación termina cuando el acero más lento la alcanza. La ✕ marca el punto frío.

In [ ]:
# >>> CAMBIO 3: animación (ajusta frames para más/menos fluidez y smooth_factor para más/menos suavizado)
heatmap_fig = comparison.plot_heatmaps(target_c, frames=40, smooth_factor=6)
heatmap_fig.show()